# Demosaic Filter Explorer
Compare all Bayer patterns × demosaic methods on your CR3 burst.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

from pipeline.raw_loader import load_raw_bayer, make_synthetic_burst
from pipeline.isp import demosaic, detect_bayer_pattern, _BAYER_ALIASES

## 1. Load image
Set `CR3_PATH` to your file, or leave `USE_DEMO = True` to use synthetic data.

In [ ]:
USE_DEMO  = True          # ← set False and fill in CR3_PATH for real data
CR3_PATH  = "data/scene_01/IMG_0001.CR3"

if USE_DEMO:
    frames = make_synthetic_burst(512, 512, n_frames=1, noise_sigma=0.03)
    bayer  = frames[0]
    reported_pattern = "RGGB (synthetic)"
    print("Using synthetic Bayer image  512×512")
else:
    bayer, reported_pattern = load_raw_bayer(CR3_PATH)
    print(f"Loaded {CR3_PATH}")
    print(f"Reported pattern : {reported_pattern}")
    print(f"Bayer shape      : {bayer.shape}")

detected = detect_bayer_pattern(bayer)
print(f"Auto-detected    : {detected}")

## 2. Raw Bayer mosaic preview

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Full mosaic (false-colour)
axes[0].imshow(bayer, cmap='gray', vmin=0, vmax=1)
axes[0].set_title("Raw Bayer mosaic (grayscale)")
axes[0].axis('off')

# Zoom into top-left 32×32 pixels to see the pattern
zoom = bayer[:32, :32]
im = axes[1].imshow(zoom, cmap='gray', interpolation='nearest')
axes[1].set_title("Top-left 32×32 px (mosaic pattern visible)")
# Overlay R/G/B labels on the 2×2 unit cell
labels = {
    'RGGB': [('R','red',(0.5,0.5)), ('G','green',(1.5,0.5)),
              ('G','green',(0.5,1.5)), ('B','blue',(1.5,1.5))],
    'BGGR': [('B','blue',(0.5,0.5)), ('G','green',(1.5,0.5)),
              ('G','green',(0.5,1.5)), ('R','red',(1.5,1.5))],
    'GRBG': [('G','green',(0.5,0.5)), ('R','red',(1.5,0.5)),
              ('B','blue',(0.5,1.5)), ('G','green',(1.5,1.5))],
    'GBRG': [('G','green',(0.5,0.5)), ('B','blue',(1.5,0.5)),
              ('R','red',(0.5,1.5)), ('G','green',(1.5,1.5))],
}
for lbl, color, (x, y) in labels.get(detected, []):
    axes[1].text(x, y, lbl, color=color, fontsize=10, fontweight='bold',
                 ha='center', va='center')
axes[1].axis('off')
plt.suptitle(f"Detected pattern: {detected}  |  Reported: {reported_pattern}",
             fontsize=13)
plt.tight_layout()
plt.show()

## 3. Compare all demosaic methods  ×  all Bayer patterns
Rows = demosaic method, Columns = Bayer pattern

In [ ]:
METHODS  = ["bilinear", "edgeaware", "vng"]
PATTERNS = ["RGGB", "BGGR", "GRBG", "GBRG", "RGBG"]  # RGBG = Canon alias for RGGB

fig, axes = plt.subplots(len(METHODS), len(PATTERNS),
                          figsize=(4 * len(PATTERNS), 3.5 * len(METHODS)))

for r, method in enumerate(METHODS):
    for c, pattern in enumerate(PATTERNS):
        rgb = demosaic(bayer, method=method, pattern=pattern)
        axes[r, c].imshow(np.clip(rgb, 0, 1))
        title = f"{method} / {pattern}"
        if _BAYER_ALIASES.get(pattern, pattern) == detected or pattern == detected:
            title += "  ✓"
        if pattern == "RGBG":
            title += "  (Canon)"
        axes[r, c].set_title(title, fontsize=9)
        axes[r, c].axis("off")

plt.suptitle("Demosaic method × Bayer pattern  (✓ = matches detected, Canon = alias for RGGB)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## 4. Side-by-side: methods on the detected pattern only

In [ ]:
results = {m: demosaic(bayer, method=m, pattern=detected) for m in METHODS}

fig, axes = plt.subplots(1, len(METHODS), figsize=(5 * len(METHODS), 5))
for ax, (method, rgb) in zip(axes, results.items()):
    ax.imshow(np.clip(rgb, 0, 1))
    ax.set_title(method, fontsize=12)
    ax.axis('off')

plt.suptitle(f"Pattern: {detected}  —  methods compared", fontsize=13)
plt.tight_layout()
plt.show()

## 5. Zoom into edge area — where methods differ most

In [ ]:
# Crop region — adjust these coordinates to an interesting edge in your image
Y0, Y1 = 100, 200
X0, X1 = 100, 200

fig, axes = plt.subplots(1, len(METHODS) + 1, figsize=(4 * (len(METHODS) + 1), 4))

# Show where the crop is
overview = demosaic(bayer, method='edgeaware', pattern=detected)
import matplotlib.patches as patches
axes[0].imshow(np.clip(overview, 0, 1))
rect = patches.Rectangle((X0, Y0), X1-X0, Y1-Y0,
                           linewidth=2, edgecolor='red', facecolor='none')
axes[0].add_patch(rect)
axes[0].set_title("Full image (crop in red)", fontsize=10)
axes[0].axis('off')

for ax, (method, rgb) in zip(axes[1:], results.items()):
    crop = rgb[Y0:Y1, X0:X1]
    ax.imshow(np.clip(crop, 0, 1), interpolation='nearest')
    ax.set_title(f"{method}\n[{Y0}:{Y1}, {X0}:{X1}]", fontsize=9)
    ax.axis('off')

plt.suptitle("Zoomed crop — demosaic artefact comparison", fontsize=12)
plt.tight_layout()
plt.show()

## 6. Channel histograms per method

In [ ]:
fig, axes = plt.subplots(1, len(METHODS), figsize=(5 * len(METHODS), 3.5))
colors = ['red', 'green', 'blue']

for ax, (method, rgb) in zip(axes, results.items()):
    for ch, (name, color) in enumerate(zip('RGB', colors)):
        vals = rgb[..., ch].ravel()
        ax.hist(vals, bins=128, range=(0, 1), color=color,
                alpha=0.5, label=name, density=True)
    ax.set_title(method, fontsize=11)
    ax.set_xlabel("Pixel value")
    ax.legend(fontsize=8)
    ax.set_xlim(0, 1)

plt.suptitle("R/G/B channel histograms by demosaic method", fontsize=12)
plt.tight_layout()
plt.show()

## 7. Pick your best method — run the full ISP

In [ ]:
from pipeline.isp import run_isp

CHOSEN_METHOD  = "edgeaware"   # ← change this after looking at the plots above
CHOSEN_PATTERN = detected      # ← or set manually, e.g. "RGGB"

rgb_u8 = run_isp(
    bayer,
    demosaic_method=CHOSEN_METHOD,
    bayer_pattern=CHOSEN_PATTERN,
    color_space="srgb",
    saturation=1.2,
    tone_map=True,
    sharpen=True,
)

plt.figure(figsize=(10, 7))
plt.imshow(rgb_u8)
plt.title(f"Full ISP output  |  {CHOSEN_METHOD} / {CHOSEN_PATTERN}", fontsize=13)
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"Output shape: {rgb_u8.shape}  dtype: {rgb_u8.dtype}")

## 8. Save chosen result

In [ ]:
import cv2
out_path = f"results/demosaic_{CHOSEN_METHOD}_{CHOSEN_PATTERN}.png"
Path("results").mkdir(exist_ok=True)
cv2.imwrite(out_path, cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2BGR))
print(f"Saved -> {out_path}")